# AI Video Factory — Automated Kaggle Free GPU Runner (₹0 / $0)

**Required Kaggle Settings / Secrets (for unattended GitHub Actions -> Kaggle runs):**
- Accelerator: `GPU T4 x2` (configured in `kernel-metadata.json`)
- Internet: `On`
- Kaggle Secrets:
  - `GITHUB_REPO_URL` (e.g. `https://github.com/your-user/ai-video-factory.git`)
  - `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_REFRESH_TOKEN` (for Drive upload)
  - `CF_WORKER_URL`, `CF_API_KEY` (optional, for Cloudflare D1 job queue)


In [ ]:
import os
import subprocess
from pathlib import Path

os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ['GITHUB_REPO_URL', 'GDRIVE_CLIENT_ID', 'GDRIVE_CLIENT_SECRET', 'GDRIVE_REFRESH_TOKEN', 'CF_WORKER_URL', 'CF_API_KEY']:
        try:
            val = secrets.get_secret(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    print('kaggle_secrets not available (local/non-Kaggle environment).')

# When pushed via `kaggle kernels push`, clone repo into /tmp/ai-video-factory
if not Path('scripts/setup.sh').exists():
    repo_url = os.environ.get('GITHUB_REPO_URL', '')
    if not repo_url:
        raise RuntimeError('Set GITHUB_REPO_URL in Kaggle Secrets so remote runs can clone the repo.')
    target_dir = Path('/tmp/ai-video-factory')
    if not target_dir.exists():
        subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(target_dir)], check=True)
    os.chdir(target_dir)
print('Working directory:', os.getcwd())

In [ ]:
!bash scripts/setup.sh
!pytest -q tests/test_gpu.py

## Milestone 1 Check: Single AI Video Clip Benchmark (Optional / First Run)

In [ ]:
# Run Milestone 1 benchmark if no benchmark output exists yet
if not Path('/kaggle/working/milestone1_clip.mp4').exists():
    !python scripts/benchmark.py --output /kaggle/working/milestone1_clip.mp4

## Automated Production Pipeline (Polls Cloudflare D1 or Uses Fallback Topic)

In [ ]:
import requests
from orchestration.pipeline_runner import run_pipeline

job_id = 'exp_001'
topic = 'How Deep Ocean Hydrothermal Vents Support Life Without Sunlight'
duration_sec = 600
num_shorts = 3

cf_url = os.environ.get('CF_WORKER_URL', '').rstrip('/')
if cf_url:
    headers = {'Authorization': f"Bearer {os.environ.get('CF_API_KEY', '')}"}
    resp = requests.get(f'{cf_url}/jobs/next', headers=headers, timeout=15)
    resp.raise_for_status()
    job = resp.json().get('job')
    if job:
        job_id = job['job_id']
        topic = job['topic']
        duration_sec = int(job.get('target_duration_sec', 600))
        num_shorts = int(job.get('num_shorts', 3))
        print(f'Picked up queued job from Cloudflare D1: {job_id} — {topic}')
    else:
        print('No QUEUED/PAUSED job in Cloudflare D1; running default experimental topic.')

result = run_pipeline(
    job_id=job_id,
    topic=topic,
    workspace_dir=Path('/kaggle/working/outputs'),
    target_duration_sec=duration_sec,
    num_shorts=num_shorts,
    upload_to_drive=bool(os.environ.get('GDRIVE_REFRESH_TOKEN')),
)
print('Pipeline finished with state:', result['state'])